# 01. Source Inventory & Coverage Scoping
## Corporate Carbon Footprint Tracker (Task EDA-1)
### Deliverable 1: Data Collection, Extraction, Cleaning & Documentation

---

### Purpose
This notebook defines the corporate cohort, sector classifications, SEC EDGAR Central Index Keys (CIKs), and public disclosure report URLs for **Deliverable 1**.
It establishes the traceable registry for downstream API collection (`02_api_collection.ipynb`) and programmatic PDF extraction (`03_pdf_extraction.ipynb`).

### Scope & Criteria
- **Reporting Period**: FY2019 – FY2023 (5-year historical window).
- **Target Entities**: 12 high-impact corporations across Technology, Energy, Industrial/Automotive, and Consumer sectors.
- **Inclusion Standards**:
  1. Mandatory SEC 10-K regulatory filings for audited financial revenues.
  2. Public standalone Sustainability / ESG / Environmental Progress reports disclosing Scope 1, Scope 2 (market & location), and Scope 3 emissions.
  3. Auditable primary documents with third-party assurance statements (e.g., Deloitte, PwC, Apex).


In [11]:
import os
import pandas as pd
import numpy as np

# Verify paths
RAW_DATA_DIR = os.path.join("..", "data", "raw")
INVENTORY_CSV = os.path.join(RAW_DATA_DIR, "source_inventory.csv")

print(f"Loading inventory from: {INVENTORY_CSV}")
assert os.path.exists(INVENTORY_CSV), f"Missing inventory file: {INVENTORY_CSV}"


Loading inventory from: ..\data\raw\source_inventory.csv


## 1. Load Corporate Cohort & Disclosures Registry

In [12]:
df_inventory = pd.read_csv(INVENTORY_CSV)
print(f"Total Companies Registered: {len(df_inventory)}")
df_inventory[['company_id', 'company_name', 'ticker', 'cik', 'sector', 'reporting_years_covered']]


Total Companies Registered: 22


,company_id,company_name,ticker,cik,sector,reporting_years_covered
0,TECH_MSFT,Microsoft Corporation,MSFT,789019,Technology,2019-2023
1,TECH_AAPL,Apple Inc.,AAPL,320193,Technology,2019-2023
2,TECH_GOOGL,Alphabet Inc.,GOOGL,1652044,Technology,2019-2023
3,TECH_AMZN,Amazon.com Inc.,AMZN,1018724,Technology / Retail,2019-2023
4,TECH_META,Meta Platforms Inc.,META,1326801,Technology,2019-2023
5,TECH_INTC,Intel Corporation,INTC,50863,Technology,2019-2023
6,TECH_NVDA,NVIDIA Corporation,NVDA,1045810,Technology,2019-2023
7,AUTO_GM,General Motors Company,GM,1467858,Automotive,2020-2023
8,ENG_XOM,ExxonMobil Corporation,XOM,34088,Energy,2019-2023
9,ENG_CVX,Chevron Corporation,CVX,93410,Energy,2019-2023


## 2. Sectoral Breakdown
Evaluating the distribution of target companies across key economic sectors to ensure cross-industry analytical validity.


In [13]:
sector_counts = df_inventory['sector'].value_counts().reset_index()
sector_counts.columns = ['Sector', 'Company Count']
print("Sector Distribution:")
display(sector_counts)


Sector Distribution:


,Sector,Company Count
0,Technology,8
1,Automotive,3
2,Energy,3
3,Healthcare,2
4,Industrials,2
5,Technology / Retail,1
6,Consumer Staples,1
7,Consumer Discretionary,1
8,Retail & Consumer,1


## 3. Coverage Matrix: Company × Reporting Years
Constructing the target observation matrix across the 5-year timeline (2019–2023).


In [14]:
years = [2019, 2020, 2021, 2022, 2023]
matrix_data = []

for _, row in df_inventory.iterrows():
    for y in years:
        matrix_data.append({
            'company_id': row['company_id'],
            'company_name': row['company_name'],
            'ticker': row['ticker'],
            'sector': row['sector'],
            'reporting_year': y,
            'status': 'Planned Extraction'
        })

df_matrix = pd.DataFrame(matrix_data)
print(f"Total Planned Company-Year Observations: {len(df_matrix)}")
pivot_summary = df_matrix.pivot(index=['company_id', 'ticker'], columns='reporting_year', values='status')
display(pivot_summary)


Total Planned Company-Year Observations: 110


,reporting_year,2019,2020,2021,2022,2023
company_id,ticker,,,,,
AUTO_F,F,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
AUTO_GM,GM,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
AUTO_TSLA,TSLA,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
CONS_KO,KO,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
CONS_NKE,NKE,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
ENG_COP,COP,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
ENG_CVX,CVX,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
ENG_XOM,XOM,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction
HLTH_JNJ,JNJ,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction,Planned Extraction


## 4. Source URL & Retrieval Protocol Verification
Documenting the access protocols, endpoints, and third-party assurance notes for every source.


In [15]:
for idx, row in df_inventory.iterrows():
    print(f"[{row['company_id']}] {row['company_name']} ({row['ticker']})")
    print(f"  Sector: {row['sector']} | CIK: {row['cik']}")
    print(f"  SEC Endpoint: https://data.sec.gov/api/xbrl/companyfacts/CIK{str(row['cik']).zfill(10)}.json")
    print(f"  Primary ESG Report: {row['primary_report_title']}")
    print(f"  Report URL: {row['report_url'][:80]}...")
    print(f"  Assurance / Notes: {row['access_notes']}")
    print("-" * 80)


[TECH_MSFT] Microsoft Corporation (MSFT)
  Sector: Technology | CIK: 789019
  SEC Endpoint: https://data.sec.gov/api/xbrl/companyfacts/CIK0000789019.json
  Primary ESG Report: Microsoft Environmental Sustainability Data Fact Sheet
  Report URL: https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/msc/docu...
  Assurance / Notes: Public disclosure; includes Deloitte & Touche third-party assurance
--------------------------------------------------------------------------------
[TECH_AAPL] Apple Inc. (AAPL)
  Sector: Technology | CIK: 320193
  SEC Endpoint: https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json
  Primary ESG Report: Apple Environmental Progress Report 2024
  Report URL: https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2024.p...
  Assurance / Notes: Public disclosure; multi-year Scope 1-3 GHG tables with Apex assurance
--------------------------------------------------------------------------------
[TECH_GOOGL] Alphabet In

## 5. Summary & Next Steps
- **Registry Established**: 12 companies across 4 major sectors.
- **Total Expected Observations**: 60 company-year records.
- **Handoff**:
  - Proceed to **`02_api_collection.ipynb`** to query the SEC EDGAR API for audited 10-K revenues.
  - Proceed to **`03_pdf_extraction.ipynb`** to download primary ESG PDFs and parse Scope 1–3 tables.
